# Search your m12 subset
Use this tool if
- You wish to start a new search of parameter space (new fixed params)
- You don't have idea for $m_12$ range of values and wish to have the approximation for power search

In [1]:
# == importar la libreria hecha ==
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), ".."))) # Añadir la ruta absoluta a 'mlpython/' al comienzo de sys.path
#import lib  

from lib import get_data_files, collect_all_rows
from lib.oracle import OracleExecutor , run_oracle

In [2]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import pdist, squareform
from matplotlib.colors import LogNorm
import json


# ---------------------
# ==== LAMBDA1 FORMULA AND SCALAR WRAPPER ====
def lambda1_C_formula(mh, mH, m12_2, sin_ba, tan_beta, lam6, lam7):
    VEV = 246.0
    tb = tan_beta
    inv = 1.0 / np.sqrt(1.0 + tb**2)
    cb = inv
    sb = tb * inv
    cba = np.sqrt(1.0 - sin_ba**2)
    ca = cb * cba + sb * sin_ba
    sa = sb * cba - cb * sin_ba

    term1 = (mH**2 * ca**2 + mh**2 * sa**2) / (VEV**2 * cb**2)
    term2 = 1.5 * lam6 * tb
    term3 = (m12_2 / VEV**2) * tb / (cb**2)
    term4 = 0.5 * lam7 * tb**3
    return term1 - term2 - term3 + term4



In [ ]:
# ==== FIXED PARAMETERS ====
mA = 300.0
sin_ba = 1.0
tan_beta = 1e3
lambda6 = 0.1
lambda7 = 0.0

m12_min, m12_max = [0, 100]

N_mphi = 100
N_m12 = 50000


model_name = "m_tan1e"+f"{int(np.log10(tan_beta))}"+"mA"+f"{mA}"
MODE = "lambda1C"

mphi_list = np.linspace(150.0, 300.0, N_mphi)

def f_scalar_mphi(m12, m_phi, mode=MODE):
    m12_val = np.asarray(m12).reshape(-1)[0]

    if mode == "lambda1C":
        return np.abs(lambda1_C_formula(
            mh=125, mH=m_phi, m12_2=m12_val,
            sin_ba=sin_ba, tan_beta=tan_beta,
            lam6=lambda6, lam7=lambda7
        ))

    elif mode == "oracle":
        oracle = OracleExecutor()
        out = oracle.run([m_phi, mA, sin_ba, tan_beta, lambda6, lambda7, m12_val])
        return np.abs(out["lambda1"])

    else:
        raise ValueError(f"Unsupported mode: {mode}")



# ==== ADAPTIVE MINIMIZATION FUNCTION ====
def run_m12_scan(mphi_list, f_scalar, bounds, tol=1e-7, maxiter=100):
    records = []
    for mphi in mphi_list:
        obj = lambda m12: f_scalar(m12, mphi)
        res = minimize_scalar(
            obj,
            bounds=bounds,
            method='bounded',
            options={'xatol': tol, 'maxiter': maxiter}
        )
        records.append((mphi, float(res.x), float(res.fun)))
    return np.array(records, dtype=[('mphi', float), ('m12_opt', float), ('f_min', float)])



# ==== CHECK FOR COLLAPSE AND REFINEMENT ====
def resolve_collapse(minima_arr, mphi_list, f_scalar,
                     delta=0.01, threshold=0.001, tol_min_width=1e-6):
    m12_opt = minima_arr['m12_opt']
    spread = np.ptp(m12_opt)
    mean_val = np.mean(m12_opt)
    range_total = m12_max - m12_min
    threshold_abs = threshold * range_total
    eps = 1e-6

    # 1) Si el óptimo está en un límite inicial, no refinamos
    if abs(min(m12_opt) - m12_min) < eps or abs(max(m12_opt) - m12_max) < eps:
        print("⚠️ Óptimo en límite detectado; no se aplica refinamiento.")
        return minima_arr, m12_min, m12_max

    # 2) Si no hay colapso (spread razonable), devolvemos original
    if spread >= threshold_abs:
        print("✅ Spread normal:", spread)
        return minima_arr, m12_min, m12_max

    # 3) Colapso real: aplicamos zoom-out controlado
    print("⚠️ Colapso detectado. Spread:", spread)
    # Expansión alrededor de la media
    delta_range = max(spread * 2, threshold_abs)
    lo = max(m12_min, mean_val - delta_range)
    hi = min(m12_max, mean_val + delta_range)

    # Validar mínimo ancho de banda
    if hi - lo < tol_min_width:
        print("❌ Intervalo demasiado pequeño tras zoom-out; usando rango original.")
        return minima_arr, m12_min, m12_max

    print(f"🔄 Zoom-Out: [{lo:.6f}, {hi:.6f}]")
    refined = run_m12_scan(mphi_list, f_scalar, (lo, hi),
                           tol=1e-10, maxiter=400)
    new_spread = np.ptp(refined['m12_opt'])
    if new_spread >= 0.01 * (hi - lo):
        print("✅ Zoom-Out exitoso, collapse resuelto.")
        return refined, lo, hi
    else:
        print("❌ No se resolvió el collapse tras refinamiento; usando rango original.")
        return minima_arr, m12_min, m12_max



# ==== INITIAL SCAN ====

bounds = (m12_min, m12_max)
minima_arr = run_m12_scan(mphi_list, f_scalar_mphi, bounds)


# ---

minima_arr, m12_min, m12_max = resolve_collapse(minima_arr, mphi_list, f_scalar_mphi)

# ==== FIT QUADRATIC MODEL ====
minima_dict = { row['mphi']: row['m12_opt'] for row in minima_arr }
x_vals = minima_arr['mphi']
y_vals = minima_arr['m12_opt']
coeffs = np.polyfit(x_vals, y_vals, deg=2)
quadratic_model = {"a": float(coeffs[0]), "b": float(coeffs[1]), "c": float(coeffs[2])}
# ==== EXPORT TO JSON ====
out_json = {
    "quadratic_model": quadratic_model,
    "search_settings": {
        "N_mphi": int(N_mphi),
        "N_m12": int(N_m12),
        "m12_min": float(m12_min),
        "m12_max": float(m12_max)
    },
    "fixed_parameters": {
        "mA": float(mA),
        "sin_ba": float(sin_ba),
        "tan_beta": float(tan_beta),
        "lambda6": float(lambda6),
        "lambda7": float(lambda7)
    }
}

with open(f"../../models/m12_quadratics/{model_name}.json", "w") as f:
    json.dump(out_json, f, indent=2)

# ==== PLOT ====
n_phi = N_mphi
n_m12 = N_m12
phi_vals = np.linspace(mphi_list.min(), mphi_list.max(), n_phi)
m12_vals = np.linspace(m12_min, m12_max, n_m12)
Phi, M12 = np.meshgrid(phi_vals, m12_vals, indexing='xy')
Phi_flat = Phi.ravel()
M12_flat = M12.ravel()
F_flat = np.array([f_scalar_mphi(m12, mphi) for m12, mphi in zip(M12_flat, Phi_flat)])
F = F_flat.reshape((n_m12, n_phi))

# ==== Compute fitted curve ====
mphi_dense = np.linspace(mphi_list.min(), mphi_list.max(), 300)
m12_fit = coeffs[0]*mphi_dense**2 + coeffs[1]*mphi_dense + coeffs[2]


plt.figure(figsize=(8, 6))
pcm = plt.pcolormesh(
    phi_vals, m12_vals, F,
    norm=LogNorm(vmin=np.nanmax(F)*1e-8, vmax=np.nanmax(F)),
    cmap='viridis', shading='auto')
plt.colorbar(pcm, label=r"$\lambda_1(m_{12}^2,\,m_\phi)$")
plt.scatter(minima_arr['mphi'], minima_arr['m12_opt'], c='red', s=40, edgecolor='white')

plt.plot(mphi_dense, m12_fit, color='white', linewidth=2, label="quadratic fit")


plt.xlabel(r"$m_\phi$ [GeV]")
plt.ylabel(r"$m_{12}^2$")
plt.title(r"Map of $\lambda_1(m_{12}^2, m_\phi)$ and local minima" + f"\nModel: {model_name}, Mode: {MODE}")
plt.legend(loc="lower right")

plt.tight_layout()
plt.savefig(f"../../models/m12_quadratics/lam1_{model_name}.png", dpi=300)
plt.show()


✅ Spread normal: 67.49993242955048


Let $y = m_{12}^2$
$$
y(m_\phi) = a m_\phi^2  + b m_\phi + c
$$

In [4]:
quadratic_model

{'a': -5.809574769279451e-17,
 'b': 2.4609603897478255e-14,
 'c': 999.9999849124076}